In [ ]:
# --- 1. Data Preprocessing (CSV to YOLO Format) ---
# Parses the raw CSV annotations, normalizes coordinates to YOLO format (0-1),
# and organizes images/labels into the required directory structure.

import pandas as pd
import shutil
import cv2
import os
from tqdm.notebook import tqdm
from ultralytics import YOLO

# Define directory structure
BASE_DIR = "yolo_dataset"
for split in ['train', 'val']:
    os.makedirs(f"{BASE_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{BASE_DIR}/labels/{split}", exist_ok=True)

def convert_to_yolo_format(csv_path, split_name):
    print(f"Processing {split_name} dataset...")
    df = pd.read_csv(csv_path)
    unique_images = df['frame'].unique()

    for img_name in tqdm(unique_images):
        src_img_path = f"self-driving-cars/images/{img_name}"
        if not os.path.exists(src_img_path): continue

        # 1. Read Image (to get dimensions)
        img = cv2.imread(src_img_path)
        h, w, _ = img.shape

        # 2. Copy Image to YOLO Directory
        dst_img_path = f"{BASE_DIR}/images/{split_name}/{img_name}"
        shutil.copy(src_img_path, dst_img_path)

        # 3. Generate YOLO Label (.txt)
        subset = df[df['frame'] == img_name]
        txt_content = ""

        for _, row in subset.iterrows():
            cls_id = row['class_id'] - 1  # Convert 1-based index to 0-based

            # Normalize Coordinates: XYXY (Pixels) -> XYWH (0-1)
            xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']

            x_center = ((xmin + xmax) / 2) / w
            y_center = ((ymin + ymax) / 2) / h
            width = (xmax - xmin) / w
            height = (ymax - ymin) / h

            txt_content += f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n"

        # Save label file
        txt_filename = img_name.replace('.jpg', '.txt').replace('.png', '.txt')
        with open(f"{BASE_DIR}/labels/{split_name}/{txt_filename}", "w") as f:
            f.write(txt_content)

# Execute conversion
convert_to_yolo_format('self-driving-cars/labels_train.csv', 'train')
convert_to_yolo_format('self-driving-cars/labels_val.csv', 'val')
print("Data preprocessing complete.")

In [ ]:
# --- 2. Create Configuration File (data.yaml) ---
# Defines the dataset path, number of classes, and class names for YOLOv8.

yaml_content = f"""
path: /content/yolo_dataset
train: images/train
val: images/val

nc: 5
names: ['car', 'truck', 'pedestrian', 'bicyclist', 'light']
"""

with open("data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created successfully.")

In [ ]:
# --- 3. Train the Model ---
# Loads the YOLOv8 Nano model and fine-tunes it on the custom dataset.
# Training automatically utilizes the GPU if available.

model = YOLO("yolov8n.pt")  # Load pre-trained Nano model

results = model.train(
    data="data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    name="self_driving_model"
)

In [ ]:
# --- 4. Export Trained Model ---
# Downloads the model weights with the best performance (best.pt) to your local machine.

from google.colab import files

try:
    files.download('runs/detect/self_driving_model/weights/best.pt')
except Exception as e:
    print(f"Error downloading file: {e}")